In [ ]:
import pandas as pd
from statsmodels.tsa.arima.model import ARIMA
from sklearn.metrics import mean_absolute_error, mean_squared_error
from scipy.stats import poisson
import numpy as np
import joblib
from datetime import datetime, timedelta
import random
import os

In [ ]:
df = pd.read_csv("road_accident.csv", encoding='latin1')

In [ ]:
df['DATE ENCODED'] = pd.to_datetime(df['DATE ENCODED'], errors='coerce')
df = df.dropna(subset=['DATE ENCODED'])

In [ ]:
daily_incidents = df.groupby(df['DATE ENCODED'].dt.date).size()
daily_incidents.index = pd.to_datetime(daily_incidents.index)
daily_incidents = daily_incidents.sort_index()


In [ ]:
print(f"Data range: {daily_incidents.index[0].date()} → {daily_incidents.index[-1].date()}")

Data range: 2022-01-01 → 2023-12-01


In [ ]:
print("\nTraining ARIMA model...")
model = ARIMA(daily_incidents, order=(2,1,2))
model_fit = model.fit()


Training ARIMA model...


/usr/local/lib/python3.12/dist-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: A date index has been provided, but it has no associated frequency information and so will be ignored when e.g. forecasting.
  self._init_dates(dates, freq)
/usr/local/lib/python3.12/dist-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: A date index has been provided, but it has no associated frequency information and so will be ignored when e.g. forecasting.
  self._init_dates(dates, freq)
/usr/local/lib/python3.12/dist-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: A date index has been provided, but it has no associated frequency information and so will be ignored when e.g. forecasting.
  self._init_dates(dates, freq)


In [ ]:
total_size = len(daily_incidents)
train_size = int(total_size * 0.7)

train = daily_incidents[:train_size]
test = daily_incidents[train_size:]

In [ ]:
model = ARIMA(train, order=(2,1,2))
model_fit = model.fit()
forecast = model_fit.forecast(steps=len(test))

/usr/local/lib/python3.12/dist-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: A date index has been provided, but it has no associated frequency information and so will be ignored when e.g. forecasting.
  self._init_dates(dates, freq)
/usr/local/lib/python3.12/dist-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: A date index has been provided, but it has no associated frequency information and so will be ignored when e.g. forecasting.
  self._init_dates(dates, freq)
/usr/local/lib/python3.12/dist-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: A date index has been provided, but it has no associated frequency information and so will be ignored when e.g. forecasting.
  self._init_dates(dates, freq)
/usr/local/lib/python3.12/dist-packages/statsmodels/tsa/base/tsa_model.py:837: ValueWarning: No supported index is available. Prediction results will be given with an integer index beginning at `start`.
  return get_prediction_index(
/usr/local/lib/p

In [ ]:
# CORRECT METRICS FOR ARIMA
mae = mean_absolute_error(test, forecast)
rmse = np.sqrt(mean_squared_error(test, forecast))
mape = np.mean(np.abs((test.values - forecast) / test.values)) * 100

In [ ]:
print(f"MAE: {mae:.2f} accidents")
print(f"RMSE: {rmse:.2f}")
print(f"MAPE: {mape:.1f}% error")

MAE: 0.81 accidents
RMSE: 0.92
MAPE: 65.0% error


In [ ]:
joblib.dump(model_fit, '/content/arima70_30.pkl')
print("Model saved: /content/arima70_30.pkl")

Model saved: /content/arima70_30.pkl


In [ ]:
# === 3. Generate RANDOM, DYNAMIC 2023 Prediction ===
def predict_2023_risk():
    # Base forecast from ARIMA
    last_date = daily_incidents.index[-1]
    days_to_end_2023 = max(1, (datetime(2023, 12, 31) - last_date).days)

    # Forecast into 2023
    base_forecast = model_fit.forecast(steps=days_to_end_2023 + 180)
    predicted_2023_total = float(base_forecast.sum())
    predicted_2023_total = max(50, predicted_2023_total)  # minimum realistic

    # === RANDOM VARIATION FOR MOVEMENT ===
    # Add natural randomness so % changes every time
    random_boost = random.uniform(0.15, 0.45)  # 15-45% extra variation
    random_noise = random.uniform(-8, 12)      # small swing

    adjusted_total = predicted_2023_total * (1 + random_boost) + random_noise
    adjusted_total = max(60, min(180, adjusted_total))  # keep realistic range

    # Convert to % risk (0-98%)
    base_prob = (adjusted_total / 150) * 100  # scale to max ~150 accidents/year
    final_prob = min(98.9, max(20.0, base_prob + random.uniform(-5, 8)))

    # Final message — clean and professional
    prediction_text = f"In 2023, there is a {final_prob:.1f}% chance another road accident will occur"

    return prediction_text, final_prob, adjusted_total

# === 4. Run prediction (test) ===
prediction, prob, count = predict_2023_risk()

/usr/local/lib/python3.12/dist-packages/statsmodels/tsa/base/tsa_model.py:837: ValueWarning: No supported index is available. Prediction results will be given with an integer index beginning at `start`.
  return get_prediction_index(


In [ ]:
print("\n" + "="*70)
print("           2023 ROAD ACCIDENT RISK FORECAST (DYNAMIC)")
print("="*70)
print(f"Forecasted accidents in 2023: ~{int(count)}")
print(f"Risk probability: {prob:.1f}%")
print("\nFINAL PREDICTION MESSAGE:")
print(prediction)
print("="*70)


           2023 ROAD ACCIDENT RISK FORECAST (DYNAMIC)
Forecasted accidents in 2023: ~180
Risk probability: 98.9%

FINAL PREDICTION MESSAGE:
In 2023, there is a 98.9% chance another road accident will occur


In [ ]:
with open("current_road_risk.txt", "w") as f:
    f.write(prediction)

# TRAINING SARIMA ⬇

In [ ]:
from statsmodels.tsa.statespace.sarimax import SARIMAX

print("\nTraining SARIMA model...")
# Define the SARIMA model order
# Non-seasonal order (p,d,q) based on the previous ARIMA model (2,1,2)
# Seasonal order (P,D,Q,s) as specified in the task (1,0,1,7)
order = (2, 1, 2)
seasonal_order = (1, 0, 1, 7)

sarima_model = SARIMAX(train, order=order, seasonal_order=seasonal_order, enforce_stationarity=False, enforce_invertibility=False)
sarima_model_fit = sarima_model.fit(disp=False)




Training SARIMA model...


/usr/local/lib/python3.12/dist-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: A date index has been provided, but it has no associated frequency information and so will be ignored when e.g. forecasting.
  self._init_dates(dates, freq)
/usr/local/lib/python3.12/dist-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: A date index has been provided, but it has no associated frequency information and so will be ignored when e.g. forecasting.
  self._init_dates(dates, freq)


In [ ]:
# Forecast on the test dataset
start_index = len(train)
end_index = len(daily_incidents) - 1
sarima_forecast = sarima_model_fit.predict(start=start_index, end=end_index)

# Ensure forecast length matches test length
# This is important because SARIMAX predict can sometimes return an extra value depending on the start/end indices
sarima_forecast = sarima_forecast[:len(test)]



/usr/local/lib/python3.12/dist-packages/statsmodels/tsa/base/tsa_model.py:837: ValueWarning: No supported index is available. Prediction results will be given with an integer index beginning at `start`.
  return get_prediction_index(
/usr/local/lib/python3.12/dist-packages/statsmodels/tsa/base/tsa_model.py:837: FutureWarning: No supported index is available. In the next version, calling this method in a model without a supported index will result in an exception.
  return get_prediction_index(


In [ ]:
# Calculate evaluation metrics
sarima_mae = mean_absolute_error(test, sarima_forecast)
sarima_rmse = np.sqrt(mean_squared_error(test, sarima_forecast))
# Handle potential division by zero or infinite values if test.values contains zeros
mape_values = np.abs((test.values - sarima_forecast) / test.values)
mape_values = mape_values[np.isfinite(mape_values)] # Filter out NaN or inf
sarima_mape = np.mean(mape_values) * 100 if len(mape_values) > 0 else 0

print(f"SARIMA MAE: {sarima_mae:.2f} accidents")
print(f"SARIMA RMSE: {sarima_rmse:.2f}")
print(f"SARIMA MAPE: {sarima_mape:.1f}% error")



SARIMA MAE: 0.77 accidents
SARIMA RMSE: 0.89
SARIMA MAPE: 63.5% error


In [ ]:
# Save the trained SARIMA model
joblib.dump(sarima_model_fit, '/content/sarima70_30.pkl')
print("Model saved: /content/sarima70_30.pkl")

Model saved: /content/sarima70_30.pkl


# Task
Train a Seasonal ARIMA (SARIMA) model with a seasonal order of (1,0,1,7) using the 'train' dataset, forecast on the 'test' dataset, calculate and print the Mean Absolute Error (MAE), Root Mean Squared Error (RMSE), and Mean Absolute Percentage Error (MAPE), and save the trained SARIMA model to '/content/sarima70_30.pkl'.

## Train and Evaluate SARIMA Model

### Subtask:
Train a Seasonal ARIMA (SARIMA) model using the 'train' dataset, forecast on the 'test' dataset, calculate evaluation metrics (MAE, RMSE, MAPE), print the results, and save the trained SARIMA model.


## Summary:

### Data Analysis Key Findings
*   A Seasonal ARIMA (SARIMA) model was trained using the `train` dataset, incorporating a seasonal order of (1,0,1,7). The non-seasonal order was established during the model specification.
*   The model's forecasting performance was evaluated on the `test` dataset, with key metrics including Mean Absolute Error (MAE), Root Mean Squared Error (RMSE), and Mean Absolute Percentage Error (MAPE) calculated and reported. (Specific metric values would be provided upon execution.)
*   The successfully trained SARIMA model was saved to `/content/sarima70_30.pkl` for later use.

### Insights or Next Steps
*   The reported MAE, RMSE, and MAPE values should be critically compared against a baseline model or other forecasting models to determine the SARIMA model's efficacy and whether it meets the desired performance criteria.
*   To further validate the model, it is recommended to perform diagnostic checks on the residuals (e.g., using Ljung-Box test or ACF/PACF plots) to ensure they are white noise, indicating that the model has captured all significant temporal dependencies.
